## Hybrid Tensor Persistence: Cloudflare + S3

This example uses both cloud pools in one workflow:
- A **large** torch tensor is uploaded to **Cloudflare R2**
- Multiple **small** torch tensors are uploaded to **AWS S3**

### Credentials
Reads credentials from:
`laila.args`


In [1]:
%load_ext autoreload
%autoreload 2


In [3]:
import laila
import torch


In [4]:
from laila.pool import S3Pool, CloudflarePool

# These argument names are arbitrary and can be changed by the user.
# laila.args is just a container for user-provided arguments.
# Replace these placeholders with your own cloud credentials before running.
laila.args.AWS_BUCKET_NAME = "your-s3-bucket-name"
laila.args.AWS_ACCESS_KEY_ID = "your-aws-access-key-id"
laila.args.AWS_SECRET_ACCESS_KEY = "your-aws-secret-access-key"
laila.args.AWS_REGION = "us-east-1"
laila.args.CLOUDFLARE_ACCOUNT_ID = "your-account-id"
laila.args.CLOUDFLARE_ACCESS_KEY_ID = "your-cloudflare-access-key-id"
laila.args.CLOUDFLARE_SECRET_ACCESS_KEY = "your-cloudflare-secret-access-key"
laila.args.CLOUDFLARE_BUCKET_NAME = "your-r2-bucket-name"

s3_pool = S3Pool(
    bucket_name=laila.args.AWS_BUCKET_NAME,
    access_key_id=laila.args.AWS_ACCESS_KEY_ID,
    secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
    region_name=laila.args.AWS_REGION,
)

cloudflare_pool = CloudflarePool(
    account_id=laila.args.CLOUDFLARE_ACCOUNT_ID,
    access_key_id=laila.args.CLOUDFLARE_ACCESS_KEY_ID,
    secret_access_key=laila.args.CLOUDFLARE_SECRET_ACCESS_KEY,
    bucket_name=laila.args.CLOUDFLARE_BUCKET_NAME,
)


In [5]:
laila.add_pool(s3_pool, pool_nickname="hybrid_s3_pool")
laila.add_pool(cloudflare_pool, pool_nickname="hybrid_cloudflare_pool")


In [6]:
# Large tensor for Cloudflare (R2)
# ~67 MB in float32 (4096 x 4096)
large_tensor = torch.randn(4096, 4096, dtype=torch.float32)
large_entry = laila.constant(data=large_tensor)

print("Large tensor shape:", tuple(large_tensor.shape))
print("Large tensor global_id:", large_entry.global_id)


Large tensor shape: (4096, 4096)
Large tensor global_id: LAILA:ENTRY:GLOBAL_ID:2ad9b6bc-1a78-48da-9164-02917c672288


In [7]:
# Multiple small tensors for S3
small_entries = [
    laila.constant(data=torch.randn(64, 64, dtype=torch.float32))
    for _ in range(12)
]

print("Small tensor entries:", len(small_entries))
print("First small tensor global_id:", small_entries[0].global_id)


Small tensor entries: 12
First small tensor global_id: LAILA:ENTRY:GLOBAL_ID:5d480e2d-7dc0-4f74-9eba-2a710b5bd18b


In [8]:
# Upload large tensor to Cloudflare
cf_future = laila.memorize(large_entry, pool_nickname="hybrid_cloudflare_pool")
print("Cloudflare future status before wait:", cf_future.status)
cf_future.wait()
print("Cloudflare future status after wait:", cf_future.status)


Cloudflare future status before wait: FutureStatus.NOT_STARTED
Cloudflare future status after wait: FutureStatus.FINISHED


In [9]:
# Upload all small tensors together to S3 (GroupFuture)
s3_group_future = laila.memorize(small_entries, pool_nickname="hybrid_s3_pool")
print("S3 future type:", type(s3_group_future).__name__)
print("S3 status before wait:", s3_group_future.status)
s3_group_future.wait()
print("S3 status after wait:", s3_group_future.status)
print("S3 child futures:", len(s3_group_future))


S3 future type: GroupFuture
S3 status before wait: {'total': 12.0, 'percentages': {'finished': 0.0, 'running': 0.0, 'not_started': 100.0, 'error': 0.0, 'cancelled': 0.0}}
S3 status after wait: {'total': 12.0, 'percentages': {'finished': 100.0, 'running': 0.0, 'not_started': 0.0, 'error': 0.0, 'cancelled': 0.0}}
S3 child futures: 12


In [10]:
# Quick verification: remember one from each pool
cf_recover_future = laila.remember(large_entry.global_id, pool_nickname="hybrid_cloudflare_pool")
cf_recover_future.wait()
recovered_large = cf_recover_future.result
assert torch.equal(recovered_large.data, large_tensor)

sample_small_entry = small_entries[0]
s3_recover_future = laila.remember(sample_small_entry.global_id, pool_nickname="hybrid_s3_pool")
s3_recover_future.wait()
recovered_small = s3_recover_future.result
assert torch.equal(recovered_small.data, sample_small_entry.data)

print("Recovery checks passed for Cloudflare and S3.")


Recovery checks passed for Cloudflare and S3.


In [11]:
# Optional cleanup
cleanup_cf = laila.forget(large_entry.global_id, pool_nickname="hybrid_cloudflare_pool")
cleanup_cf.wait()

cleanup_s3 = laila.forget([e.global_id for e in small_entries], pool_nickname="hybrid_s3_pool")
cleanup_s3.wait()

print("Cleanup complete.")


Cleanup complete.
